In [28]:
using LowLevelFEM, LinearAlgebra

In [29]:
using SparseArrays

"""
    reductionMatrices(P::Problem)

Construct sparse transformation matrices for reducing the polynomial order
of a C0 Lagrange finite element field from order `p` to `p - 1`.

The returned matrices satisfy

    u_full = T * u_reduced
    u_reduced = R * u_full

for fields representable in the reduced space.

The transformation is constructed from the Gmsh Lagrange basis functions.
All element types belonging to the problem must have the same polynomial
order. An error is thrown for first-order meshes.

The matrices are expanded automatically according to `P.pdim`.
"""
function reductionMatrices(P::Problem)

    gmsh.model.setCurrent(P.name)

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    function element_family(name::String)
        if occursin("Line", name)
            return "Line"
        elseif occursin("Triangle", name)
            return "Triangle"
        elseif occursin("Quadrilateral", name) ||
               occursin("Quadrangle", name)
            return "Quadrangle"
        elseif occursin("Tetrahedron", name)
            return "Tetrahedron"
        elseif occursin("Hexahedron", name)
            return "Hexahedron"
        elseif occursin("Prism", name)
            return "Prism"
        elseif occursin("Pyramid", name)
            return "Pyramid"
        else
            error("reductionMatrices: unsupported Gmsh element family \"$name\".")
        end
    end

    # getElementProperties returns dim coordinates/node,
    # while getBasisFunctions expects (u,v,w) triplets.
    function local_to_3d(localCoord, dim, n)
        ξ = zeros(Float64, 3*n)

        @inbounds for a in 1:n
            for d in 1:dim
                ξ[3*(a-1)+d] =
                    localCoord[dim*(a-1)+d]
            end
        end

        return ξ
    end

    # ------------------------------------------------------------------
    # Node coordinates
    # ------------------------------------------------------------------

    nodeTags, coord, _ =
        gmsh.model.mesh.getNodes(-1, -1, false, false)

    nodeCoord = Dict{UInt64,NTuple{3,Float64}}()
    sizehint!(nodeCoord, length(nodeTags))

    @inbounds for i in eachindex(nodeTags)
        nodeCoord[nodeTags[i]] = (
            coord[3*i-2],
            coord[3*i-1],
            coord[3*i]
        )
    end

    # ------------------------------------------------------------------
    # Collect all elements belonging to the Problem
    # ------------------------------------------------------------------

    elements = NamedTuple[]
    seen_elements = Set{UInt64}()
    orders = Set{Int}()

    # Cache data depending only on element type
    type_cache = Dict{Int,Any}()

    for mat in P.material

        dimTags =
            gmsh.model.getEntitiesForPhysicalName(mat.phName)

        for (edim, etag) in dimTags

            # Only domain elements are relevant here
            edim == P.dim || continue

            elemTypes, elemTags, elemNodeTags =
                gmsh.model.mesh.getElements(edim, etag)

            for it in eachindex(elemTypes)

                et = elemTypes[it]

                name, dim, p, nHigh, ξHigh0, nPrimary =
                    gmsh.model.mesh.getElementProperties(et)

                push!(orders, p)

                if !haskey(type_cache, et)

                    p > 1 ||
                        error(
                            "reductionMatrices: polynomial order must be greater than one."
                        )

                    q = p - 1
                    family = element_family(name)

                    etLow =
                        gmsh.model.mesh.getElementType(
                            family,
                            q,
                            false
                        )

                    nameLow, dimLow, qCheck,
                    nLow, ξLow0, nPrimaryLow =
                        gmsh.model.mesh.getElementProperties(etLow)

                    dimLow == dim ||
                        error(
                            "reductionMatrices: incompatible reduced element dimension."
                        )

                    qCheck == q ||
                        error(
                            "reductionMatrices: could not construct order-$q element."
                        )

                    # ----------------------------------------------
                    # Local prolongation:
                    #
                    #   u_high = Te * u_low
                    #
                    # Evaluate low-order basis at high-order nodes.
                    # ----------------------------------------------

                    ξHigh = local_to_3d(ξHigh0, dim, nHigh)

                    _, funT, _ =
                        gmsh.model.mesh.getBasisFunctions(
                            etLow,
                            ξHigh,
                            "Lagrange"
                        )

                    Te =
                        Matrix(
                            transpose(
                                reshape(funT, nLow, nHigh)
                            )
                        )

                    # ----------------------------------------------
                    # Local restriction:
                    #
                    #   u_low = Re * u_high
                    #
                    # Evaluate high-order basis at low-order nodes.
                    # ----------------------------------------------

                    ξLow = local_to_3d(ξLow0, dim, nLow)

                    _, funR, _ =
                        gmsh.model.mesh.getBasisFunctions(
                            et,
                            ξLow,
                            "Lagrange"
                        )

                    Re =
                        Matrix(
                            transpose(
                                reshape(funR, nHigh, nLow)
                            )
                        )

                    type_cache[et] = (
                        etLow=etLow,
                        p=p,
                        q=q,
                        nHigh=nHigh,
                        nLow=nLow,
                        nPrimary=nPrimary,
                        nPrimaryLow=nPrimaryLow,
                        Te=Te,
                        Re=Re
                    )
                end

                cache = type_cache[et]

                tags = elemTags[it]
                conn = elemNodeTags[it]

                @inbounds for e in eachindex(tags)

                    elemTag = tags[e]

                    elemTag in seen_elements && continue
                    push!(seen_elements, elemTag)

                    o = (e - 1) * cache.nHigh

                    nodes =
                        UInt64.(
                            conn[(o+1):(o+cache.nHigh)]
                        )

                    push!(
                        elements,
                        (
                            tag=elemTag,
                            et=et,
                            nodes=nodes,
                            primary=nodes[1:cache.nPrimary]
                        )
                    )
                end
            end
        end
    end

    isempty(elements) &&
        error("reductionMatrices: no domain elements found.")

    length(orders) == 1 ||
        error(
            "reductionMatrices: the mesh must have homogeneous polynomial order; found $(sort!(collect(orders)))."
        )

    p = first(orders)

    p > 1 ||
        error(
            "reductionMatrices: polynomial order must be greater than one."
        )

    # ------------------------------------------------------------------
    # Characteristic length for coordinate comparisons
    # ------------------------------------------------------------------

    used_nodes = Set{UInt64}()

    for elem in elements
        union!(used_nodes, elem.nodes)
    end

    xmin = Inf
    ymin = Inf
    zmin = Inf
    xmax = -Inf
    ymax = -Inf
    zmax = -Inf

    for node in used_nodes
        x, y, z = nodeCoord[node]

        xmin = min(xmin, x)
        ymin = min(ymin, y)
        zmin = min(zmin, z)

        xmax = max(xmax, x)
        ymax = max(ymax, y)
        zmax = max(zmax, z)
    end

    L = max(xmax - xmin, ymax - ymin, zmax - zmin)

    L > 0 ||
        error("reductionMatrices: degenerate mesh geometry.")

    tol = 1e-10 * L
    tol2 = tol^2

    # ------------------------------------------------------------------
    # Build reduced global node numbering
    # ------------------------------------------------------------------

    nElem = length(elements)

    reduced_conn =
        Vector{Vector{Int}}(undef, nElem)

    reduced_coord =
        Vector{Matrix{Float64}}(undef, nElem)

    # Original vertex node -> reduced node
    vertex_to_reduced =
        Dict{UInt64,Int}()

    # Original primary node -> already processed elements
    node_to_elements =
        Dict{UInt64,Vector{Int}}()

    nReduced = 0

    for ie in 1:nElem

        elem = elements[ie]
        cache = type_cache[elem.et]

        nHigh = cache.nHigh
        nLow = cache.nLow

        # ----------------------------------------------
        # Physical coordinates of high-order nodes
        # ----------------------------------------------

        Xhigh = zeros(Float64, 3, nHigh)

        @inbounds for a in 1:nHigh
            x, y, z = nodeCoord[elem.nodes[a]]

            Xhigh[1, a] = x
            Xhigh[2, a] = y
            Xhigh[3, a] = z
        end

        # Physical positions of reduced nodes:
        #
        # Xlow = Re * Xhigh'
        #
        Xlow =
            transpose(cache.Re * transpose(Xhigh))

        reduced_coord[ie] = Xlow

        rconn = zeros(Int, nLow)

        # ----------------------------------------------
        # Potential already-processed neighbours.
        #
        # Count shared primary nodes.
        # ----------------------------------------------

        neighbour_count = Dict{Int,Int}()

        for node in elem.primary
            for je in get(node_to_elements, node, Int[])
                neighbour_count[je] =
                    get(neighbour_count, je, 0) + 1
            end
        end

        neighbours =
            [
                je for (je, count) in neighbour_count
                       if count >= 2
            ]

        # ----------------------------------------------
        # Reduced nodes
        # ----------------------------------------------

        @inbounds for a in 1:nLow

            # Vertices are guaranteed to correspond to the original
            # primary nodes and can be identified exactly by node tag.
            if a <= cache.nPrimaryLow

                original_vertex = elem.nodes[a]

                if haskey(vertex_to_reduced, original_vertex)

                    rconn[a] =
                        vertex_to_reduced[original_vertex]

                else

                    nReduced += 1

                    vertex_to_reduced[original_vertex] =
                        nReduced

                    rconn[a] = nReduced
                end

                continue
            end

            # Non-vertex reduced node: search only in topologically
            # connected previously processed elements.
            xa = Xlow[1, a]
            ya = Xlow[2, a]
            za = Xlow[3, a]

            found = 0

            for je in neighbours

                Xold = reduced_coord[je]
                rold = reduced_conn[je]

                for b in axes(Xold, 2)

                    dx = xa - Xold[1, b]
                    dy = ya - Xold[2, b]
                    dz = za - Xold[3, b]

                    if dx^2 + dy^2 + dz^2 <= tol2
                        found = rold[b]
                        break
                    end
                end

                found != 0 && break
            end

            if found == 0
                nReduced += 1
                rconn[a] = nReduced
            else
                rconn[a] = found
            end
        end

        reduced_conn[ie] = rconn

        # Register this element as processed
        for node in elem.primary
            push!(
                get!(node_to_elements, node, Int[]),
                ie
            )
        end
    end

    # ------------------------------------------------------------------
    # Assemble scalar T
    # ------------------------------------------------------------------

    IT = Int[]
    JT = Int[]
    VT = Float64[]

    full_seen = falses(P.non)

    for ie in 1:nElem

        elem = elements[ie]
        cache = type_cache[elem.et]
        rconn = reduced_conn[ie]

        Te = cache.Te

        @inbounds for a in 1:cache.nHigh

            node = Int(elem.nodes[a])

            node <= P.non ||
                error(
                    "reductionMatrices: node tag $node exceeds problem.non=$(P.non)."
                )

            full_seen[node] && continue
            full_seen[node] = true

            for b in 1:cache.nLow

                v = Te[a, b]

                abs(v) < 100eps(Float64) && continue

                push!(IT, node)
                push!(JT, rconn[b])
                push!(VT, v)
            end
        end
    end

    Tn =
        sparse(
            IT,
            JT,
            VT,
            P.non,
            nReduced
        )

    # ------------------------------------------------------------------
    # Assemble scalar R
    #
    # A reduced node can belong to several elements. Its interpolation
    # row is taken from the first processed element containing it.
    # ------------------------------------------------------------------

    IR = Int[]
    JR = Int[]
    VR = Float64[]

    reduced_seen = falses(nReduced)

    for ie in 1:nElem

        elem = elements[ie]
        cache = type_cache[elem.et]
        rconn = reduced_conn[ie]

        Re = cache.Re

        @inbounds for a in 1:cache.nLow

            rnode = rconn[a]

            reduced_seen[rnode] && continue
            reduced_seen[rnode] = true

            for b in 1:cache.nHigh

                v = Re[a, b]

                abs(v) < 100eps(Float64) && continue

                push!(IR, rnode)
                push!(JR, Int(elem.nodes[b]))
                push!(VR, v)
            end
        end
    end

    Rn =
        sparse(
            IR,
            JR,
            VR,
            nReduced,
            P.non
        )

    # ------------------------------------------------------------------
    # Expand according to pdim
    #
    # DOF ordering:
    #
    # node1_comp1, node1_comp2, ...,
    # node2_comp1, ...
    # ------------------------------------------------------------------

    d = P.pdim

    Id = spdiagm(0 => ones(Float64, d))

    T = kron(Tn, Id)
    R = kron(Rn, Id)

    return T, R
end

reductionMatrices

In [30]:
using SparseArrays

"""
    q2_to_q1_transformation(P)

Build a sparse prolongation matrix that embeds a Q1 scalar field
into the Q2 nodal space of a quadrilateral mesh.

The returned matrix `T` satisfies

    p_Q2 = T * p_Q1

where Q2 edge nodes are interpolated from the two adjacent corner nodes
and the center node is interpolated from all four corner nodes.
"""
function q2_to_q1_transformation(P::Problem)

    gmsh.model.setCurrent(P.name)

    # --- collect all 2D elements belonging to the problem ---
    relations = Dict{Int,Dict{Int,Float64}}()
    primary_nodes = Set{Int}()

    function set_relation!(node, cols, vals)
        rel = Dict(Int(c) => Float64(v) for (c, v) in zip(cols, vals))

        if haskey(relations, node)
            relations[node] == rel ||
                error("Inconsistent Q2→Q1 relation for node $node.")
        else
            relations[node] = rel
        end
    end

    for mat in P.material
        dimTags = gmsh.model.getEntitiesForPhysicalName(mat.phName)

        for (edim, etag) in dimTags
            edim == 2 || continue

            elemTypes, elemTags, elemNodeTags =
                gmsh.model.mesh.getElements(edim, etag)

            for it in eachindex(elemTypes)

                et = elemTypes[it]

                _, _, order, numNodes, _, numPrimaryNodes =
                    gmsh.model.mesh.getElementProperties(et)

                order == 2 ||
                    error("q2_to_q1_transformation requires second-order elements.")

                numPrimaryNodes == 4 ||
                    error("Only quadrilateral Q2 elements are supported in this prototype.")

                numNodes == 9 ||
                    error("This prototype expects 9-node Q2 quadrilaterals.")

                conn = elemNodeTags[it]
                nel = length(elemTags[it])

                for e in 1:nel
                    o = (e - 1) * numNodes

                    n1 = Int(conn[o+1])
                    n2 = Int(conn[o+2])
                    n3 = Int(conn[o+3])
                    n4 = Int(conn[o+4])

                    n5 = Int(conn[o+5])
                    n6 = Int(conn[o+6])
                    n7 = Int(conn[o+7])
                    n8 = Int(conn[o+8])
                    n9 = Int(conn[o+9])

                    union!(primary_nodes, (n1, n2, n3, n4))

                    # corner nodes
                    set_relation!(n1, [n1], [1.0])
                    set_relation!(n2, [n2], [1.0])
                    set_relation!(n3, [n3], [1.0])
                    set_relation!(n4, [n4], [1.0])

                    # edge nodes
                    set_relation!(n5, [n1, n2], [0.5, 0.5])
                    set_relation!(n6, [n2, n3], [0.5, 0.5])
                    set_relation!(n7, [n3, n4], [0.5, 0.5])
                    set_relation!(n8, [n4, n1], [0.5, 0.5])

                    # center node
                    set_relation!(
                        n9,
                        [n1, n2, n3, n4],
                        [0.25, 0.25, 0.25, 0.25]
                    )
                end
            end
        end
    end

    primary = sort!(collect(primary_nodes))
    reduced_index = Dict(node => i for (i, node) in enumerate(primary))

    I = Int[]
    J = Int[]
    V = Float64[]

    for node in sort!(collect(keys(relations)))
        for (master, weight) in relations[node]
            push!(I, node)
            push!(J, reduced_index[master])
            push!(V, weight)
        end
    end

    T = sparse(I, J, V, P.non, length(primary))

    return T, primary
end

q2_to_q1_transformation

In [31]:
structured_rect_mesh(lx=2, order=2)

In [32]:
material = Material("body")

Pp = Problem([material], type=:ScalarField, field=:p, rhs_field=:fp, dim=2)
Pv = Problem([material], type=:VectorField, field=:v, rhs_field=:fv, dim=2);

In [33]:
pres1 = BoundaryCondition("rightbottom", problem=Pp, p=0);

In [34]:
suppT = BoundaryCondition("top", problem=Pv, vx=0, vy=0)
suppB = BoundaryCondition("bottom", problem=Pv, vx=0, vy=0);

In [35]:
load_v = LoadCondition("body", fvx=1.0, fvy=0.0)
fv = loadVector(Pv, [load_v])

gp = loadVector(Pp, [])

F = SystemVector([fv, gp]);

In [36]:
μ = 1.0

A = ∫((SymGrad(Pv) ⋅ SymGrad(Pv)) * 2μ)

B = ∫(Div(Pv) ⋅ Pp);

In [37]:
γ = 1e-1          # grad-div ( 1e-2...1e0)
δ = 1e-4          # pressure Laplacian (mesh dependent)

C = ∫(Grad(Pp) ⋅ Grad(Pp) * δ)

D = ∫(Div(Pv) ⋅ Div(Pv) * γ);

In [38]:
# alternative way
AD = ∫((SymGrad(Pv) ⋅ SymGrad(Pv)) * 2μ + γ * (Div(Pv) ⋅ Div(Pv)));

In [39]:
K = SystemMatrix([A+D B;
    B' -C])

K[:, :]

2583×2583 SparseMatrixCSC{Float64, Int64} with 116898 stored entries:
⎡⣿⢟⣉⠙⠿⠷⠶⠶⢾⡋⠯⠽⠸⠸⠶⠆⠆⠆⠶⠰⠰⠰⠶⠆⡆⣋⣻⣏⠻⠷⢾⡿⠿⠷⠶⠶⠶⠶⢶⣋⎤
⎢⣇⠘⢿⣷⡶⠾⠿⠛⠛⣷⡶⠰⠰⠶⠆⠇⠇⠭⠽⠘⠘⠛⠃⠃⠃⠛⢹⢻⣷⠿⠛⣷⠶⠶⠿⠽⠛⠛⠛⠛⎥
⎢⢿⡇⣸⡏⠻⣦⡀⠀⠀⠉⠛⠻⠶⣶⣤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣾⡿⣆⠀⠙⠷⣦⣄⠀⠀⠀⠀⠀⎥
⎢⢸⡇⣿⠃⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠈⠉⠙⠛⠷⢶⣦⣤⣀⡀⠀⠀⢸⣿⠁⠹⣆⠀⠀⠈⠙⠻⣦⣄⡀⠀⎥
⎢⡾⠳⢿⣤⡄⠀⠀⠈⠻⣦⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⠛⠻⠶⢾⢿⣤⠀⠹⣦⠀⠀⠀⠀⠀⠙⠻⠶⎥
⎢⣏⡇⢘⡋⣿⡀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣘⣻⡀⠀⢿⣧⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣒⡂⢰⡆⢸⣧⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢐⣲⣾⡇⠀⠀⢻⣧⠀⠀⠀⠀⠀⠀⎥
⎢⠸⠇⠬⠅⠀⢿⡆⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠸⠯⠅⣧⠀⠀⠀⢿⣧⠀⠀⠀⠀⠀⎥
⎢⠨⠅⡍⡅⠀⠘⣷⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⠀⠀⠨⢭⠅⢻⠀⠀⠀⠀⢻⣧⠀⠀⠀⠀⎥
⎢⢘⡃⣓⠃⠀⠀⢹⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⠀⠀⢘⣛⠀⢸⡇⠀⠀⠀⠀⢻⣧⠀⠀⠀⎥
⎢⢐⡂⣶⠀⠀⠀⠈⣿⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠀⠀⢐⣲⠀⠈⣇⠀⠀⠀⠀⠀⢻⣧⠀⠀⎥
⎢⠸⠇⠭⠀⠀⠀⠀⠸⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣄⠀⠸⠯⠀⠀⣿⠀⠀⠀⠀⠀⠈⢻⣧⠀⎥
⎢⡬⢩⣭⠀⠀⠀⠀⠀⢻⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢿⣷⣬⣭⠀⠀⢸⡆⠀⠀⠀⠀⠀⠀⢻⣧⎥
⎢⡿⢾⣷⣒⣲⣶⣶⣶⣾⣗⣒⢲⢰⣰⡶⡆⡆⣆⣶⢰⢰⣰⡶⡆⡆⣿⣿⣿⣲⣶⣾⣗⣶⣶⣶⣶⣶⣶⣶⣿⎥
⎢⢿⡆⣽⡟⠻⢯⣅⡀⠀⠛⠛⠺⠾⠿⠥⣥⣥⣁⣀⣀⡀⠀⠀⠀⠀⠀⢸⣾⡿⣯⡀⠛⠿⠯⣭⣁⣀⠀⠀⠀⎥
⎢⣾⡷⢿⣤⣄⠀⠈⠙⠳⣦⣤⣄⠀⠀⠀⠀⠀⠀⠉⠉⠉⠙⠛⠛⠲⠶⢾⢿⣤⠈⠻⣦⡀⠀⠀⠈⠉⠛⠳⠶⎥
⎢⢿⡇⢸⡇⠹⣧⡀⠀⠀⠀⠉⠛⠿⣶⣤⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⡿⡇⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⢸⡇⣟⡇⠀⠙⣷⡀⠀⠀⠀⠀⠀⠀⠉⠛⠿⣶⣤⣀⠀⠀⠀⠀⠀⠀⢸⣿⠇⢻⡀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⢸⡇⣿⠀⠀⠀⠈⢿⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠛⠿⣶⣦⣀⠀⠀⢸⣿⠀⠘⣧⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⡼⢳⣿⠀⠀⠀⠀⠈⢻⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠛⠿⣶⣼⣿⠀⠀⢹⡆⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [40]:
T, R = reductionMatrices(Pp)

println("T: ", size(T), ", nnz = ", nnz(T))
println("R: ", size(R), ", nnz = ", nnz(R))

println("RT size = ", size(R * T))
println("||RT - I||∞ = ",
    norm(R * T - I, Inf))

T: (861, 231), nnz = 1891
R: (231, 861), nnz = 231
RT size = (231, 231)
||RT - I||∞ = 0.0


In [41]:
one_full = ones(size(T, 1))
one_red = ones(size(T, 2))

println("T*1 error = ",
    maximum(abs.(T * one_red - one_full)))

println("R*1 error = ",
    maximum(abs.(R * one_full - one_red)))

T*1 error = 0.0
R*1 error = 0.0


In [49]:
Tnew, Rnew = reductionMatrices(Pp)
Told, primary = q2_to_q1_transformation(Pp)

(sparse([1, 24, 120, 294, 2, 43, 53, 844, 3, 62  …  860, 291, 838, 839, 840, 841, 842, 859, 860, 861], [1, 1, 1, 1, 2, 2, 2, 2, 3, 3  …  230, 231, 231, 231, 231, 231, 231, 231, 231, 231], [1.0, 0.5, 0.5, 0.25, 1.0, 0.5, 0.5, 0.25, 1.0, 0.5  …  0.25, 1.0, 0.5, 0.5, 0.25, 0.5, 0.25, 0.5, 0.25, 0.25], 861, 231), [1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  282, 283, 284, 285, 286, 287, 288, 289, 290, 291])

In [42]:
v, p = solveField(K, F, support=[pres1, suppT, suppB]);

In [ ]:
pnew = Tnew * p

In [43]:
showDoFResults(v, name="v", visible=true)
showDoFResults(p, name="p");

In [44]:
"""
    inspect_reduced_keys(P)

Inspect the Gmsh basis-function keys for the first element of `P`
using a Lagrange space of order `p - 1`.
"""
function inspect_reduced_keys(P::Problem)

    gmsh.model.setCurrent(P.name)

    # Find the first element belonging to the problem
    et_high = nothing
    elem_tag = nothing

    for mat in P.material
        dimTags = gmsh.model.getEntitiesForPhysicalName(mat.phName)

        for (dim, tag) in dimTags
            elementTypes, elementTags, _ =
                gmsh.model.mesh.getElements(dim, tag)

            for k in eachindex(elementTypes)
                isempty(elementTags[k]) && continue

                et_high = elementTypes[k]
                elem_tag = elementTags[k][1]
                break
            end

            et_high === nothing || break
        end

        et_high === nothing || break
    end

    et_high === nothing &&
        error("No elements found for problem $(P.name).")

    name, dim, p, n_high, ξ_high, n_primary =
        gmsh.model.mesh.getElementProperties(et_high)

    p > 1 || error("Polynomial order must be greater than one.")

    q = p - 1

    println("High-order element:")
    println("  type          = ", et_high)
    println("  name          = ", name)
    println("  dimension     = ", dim)
    println("  order         = ", p)
    println("  nodes         = ", n_high)
    println("  primary nodes = ", n_primary)
    println("  element tag   = ", elem_tag)

    # For the current quadrilateral test
    family = if occursin("Quadrilateral", name) || occursin("Quadrangle", name)
        "Quadrangle"
    elseif occursin("Triangle", name)
        "Triangle"
    elseif occursin("Tetrahedron", name)
        "Tetrahedron"
    elseif occursin("Hexahedron", name)
        "Hexahedron"
    elseif occursin("Prism", name)
        "Prism"
    elseif occursin("Pyramid", name)
        "Pyramid"
    elseif occursin("Line", name)
        "Line"
    else
        error("Unknown element family: $name")
    end

    et_low = gmsh.model.mesh.getElementType(family, q, false)

    name_low, dim_low, q_check, n_low, ξ_low, n_primary_low =
        gmsh.model.mesh.getElementProperties(et_low)

    println("\nReduced element:")
    println("  type          = ", et_low)
    println("  name          = ", name_low)
    println("  order         = ", q_check)
    println("  nodes         = ", n_low)
    println("  primary nodes = ", n_primary_low)

    # Keys of the ACTUAL high-order element, but request a q-th order
    # Lagrange function space.
    function_space = "Lagrange$(q)"

    typeKeys, entityKeys, coord =
        gmsh.model.mesh.getKeysForElement(
            elem_tag,
            function_space,
            true
        )

    println("\nFunction space = ", function_space)
    println("number of keys = ", length(typeKeys))

    println("\nkeys:")
    for i in eachindex(typeKeys)
        x = coord[3i-2]
        y = coord[3i-1]
        z = coord[3i]

        println(
            lpad(i, 3),
            ":  typeKey = ", typeKeys[i],
            ", entityKey = ", entityKeys[i],
            ", coord = (", x, ", ", y, ", ", z, ")"
        )
    end

    return (
        et_high=et_high,
        et_low=et_low,
        element_tag=elem_tag,
        typeKeys=typeKeys,
        entityKeys=entityKeys,
        coord=coord
    )
end

inspect_reduced_keys

In [45]:
"""
    inspect_shared_reduced_keys(P)

Find two elements sharing at least two primary nodes and compare their
`Lagrange(p-1)` Gmsh basis-function keys.
"""
function inspect_shared_reduced_keys(P::Problem)

    gmsh.model.setCurrent(P.name)

    elements = NamedTuple[]

    p = nothing

    for mat in P.material
        dimTags = gmsh.model.getEntitiesForPhysicalName(mat.phName)

        for (dim, tag) in dimTags
            elementTypes, elementTags, elemNodeTags =
                gmsh.model.mesh.getElements(dim, tag)

            for k in eachindex(elementTypes)
                et = elementTypes[k]

                name, _, order, nnode, _, nprimary =
                    gmsh.model.mesh.getElementProperties(et)

                if p === nothing
                    p = order
                elseif order != p
                    error("Non-homogeneous polynomial order detected.")
                end

                tags = elementTags[k]
                conn = elemNodeTags[k]

                for e in eachindex(tags)
                    o = (e - 1) * nnode

                    nodes = Int.(conn[(o+1):(o+nnode)])
                    primary = nodes[1:nprimary]

                    push!(
                        elements,
                        (
                            tag=tags[e],
                            et=et,
                            name=name,
                            nodes=nodes,
                            primary=primary
                        )
                    )
                end
            end
        end
    end

    p === nothing && error("No elements found.")
    p > 1 || error("Polynomial order must be greater than one.")

    q = p - 1
    function_space = "Lagrange$(q)"

    # Find two elements sharing an edge.
    e1 = nothing
    e2 = nothing

    for i in 1:(length(elements)-1)
        s1 = Set(elements[i].primary)

        for j in (i+1):length(elements)
            shared = intersect(s1, Set(elements[j].primary))

            if length(shared) >= 2
                e1 = elements[i]
                e2 = elements[j]
                break
            end
        end

        e1 === nothing || break
    end

    e1 === nothing &&
        error("Could not find neighbouring elements.")

    println("Element 1 = ", e1.tag)
    println("Element 2 = ", e2.tag)
    println("Shared primary nodes = ",
        intersect(Set(e1.primary), Set(e2.primary)))

    tk1, ek1, c1 =
        gmsh.model.mesh.getKeysForElement(
            e1.tag,
            function_space,
            true
        )

    tk2, ek2, c2 =
        gmsh.model.mesh.getKeysForElement(
            e2.tag,
            function_space,
            true
        )

    keys1 = Set(zip(tk1, ek1))
    keys2 = Set(zip(tk2, ek2))

    shared_keys = intersect(keys1, keys2)

    println("\nFunction space = ", function_space)
    println("keys element 1 = ", length(keys1))
    println("keys element 2 = ", length(keys2))
    println("shared keys    = ", length(shared_keys))

    println("\nShared key pairs:")
    for key in shared_keys
        println("  ", key)
    end

    return shared_keys
end

inspect_shared_reduced_keys

In [46]:
a = inspect_reduced_keys(Pp)

High-order element:
  type          = 10
  name          = Quadrilateral 9
  dimension     = 2
  order         = 2
  nodes         = 9
  primary nodes = 4
  element tag   = 65

Reduced element:
  type          = 3
  name          = Quadrilateral 4
  order         = 1
  nodes         = 4
  primary nodes = 4

Function space = Lagrange1
number of keys = 9

keys:
  1:  typeKey = 0, entityKey = 1, coord = (0.0, 0.0, 0.0)
  2:  typeKey = 0, entityKey = 5, coord = (0.1, 0.0, 0.0)
  3:  typeKey = 0, entityKey = 121, coord = (0.09999999999999987, 0.09999999999999998, 0.0)
  4:  typeKey = 0, entityKey = 110, coord = (0.0, 0.09999999999999998, 0.0)
  5:  typeKey = 0, entityKey = 24, coord = (0.05, 0.0, 0.0)
  6:  typeKey = 0, entityKey = 292, coord = (0.09999999999999987, 0.04999999999999999, 0.0)
  7:  typeKey = 0, entityKey = 293, coord = (0.04999999999999993, 0.09999999999999998, 0.0)
  8:  typeKey = 0, entityKey = 120, coord = (0.0, 0.050000000000000044, 0.0)
  9:  typeKey = 0, entityKey = 29

(et_high = 10, et_low = 3, element_tag = 0x0000000000000041, typeKeys = Int32[0, 0, 0, 0, 0, 0, 0, 0, 0], entityKeys = UInt64[0x0000000000000001, 0x0000000000000005, 0x0000000000000079, 0x000000000000006e, 0x0000000000000018, 0x0000000000000124, 0x0000000000000125, 0x0000000000000078, 0x0000000000000126], coord = [0.0, 0.0, 0.0, 0.1, 0.0, 0.0, 0.09999999999999987, 0.09999999999999998, 0.0, 0.0  …  0.0, 0.04999999999999993, 0.09999999999999998, 0.0, 0.0, 0.050000000000000044, 0.0, 0.04999999999999982, 0.04999999999999993, 0.0])

In [47]:
b = inspect_shared_reduced_keys(Pp)

Element 1 = 65
Element 2 = 66
Shared primary nodes = Set([110, 121])

Function space = Lagrange1
keys element 1 = 9
keys element 2 = 9
shared keys    = 3

Shared key pairs:
  (0, 0x000000000000006e)
  (0, 0x0000000000000125)
  (0, 0x0000000000000079)


Set{Tuple{Int32, UInt64}} with 3 elements:
  (0, 0x000000000000006e)
  (0, 0x0000000000000125)
  (0, 0x0000000000000079)

In [48]:
openPostProcessor()